# Interval Memory for Long Video — GPU IngestRuns the **ingest half** of the system on Kaggle/Colab: video frames → SigLIPtagging → selective Qwen2-VL escalation → intervals → resolved memory → acommitted **feature pack**.Everything downstream of this notebook is CPU-only. You run this once per video,download the pack, commit it, and reviewers reproduce the demo with no GPU.**Before running:** turn on the GPU accelerator (T4 is plenty) and enableInternet (needed to pull model weights from Hugging Face).

## 1. Config

In [ ]:
# --- where the source code comes from ---
# Option A: push the repo to GitHub and set REPO_URL
# Option B: upload the repo as a Kaggle Dataset and set SOURCE_DIR to it
REPO_URL   = ""                                  # e.g. "https://github.com/you/video-memory"
SOURCE_DIR = "/kaggle/input/video-memory"        # used when REPO_URL is empty

# --- the video to ingest ---
VIDEO_PATH = ""      # set to an .mp4 path; leave empty to use the synthetic fallback

# --- ingest settings ---
SAMPLE_FPS   = 1.0   # frames sampled per second of video
TAU          = 2.0   # occlusion tolerance (s) for the gap rule
MAX_FRAMES   = 300   # cap for a first smoke run; set None for the full video
MIN_DURATION = 0.0   # drop resolved fragments shorter than this
STAGE3       = True  # False = ablation A-noVLM (SigLIP only)

OUT_PACK = "/kaggle/working/pack"

## 2. Install dependencies

In [ ]:
# Qwen2-VL needs transformers >= 4.45; SigLIP is well supported there too.
!pip -q install "transformers>=4.45" accelerate opencv-python-headless
import transformers, torch
print("transformers", transformers.__version__)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 3. Get the source code

In [ ]:
import os, sys, subprocess, shutil

WORK = "/kaggle/working/video-memory"
if REPO_URL:
    if os.path.exists(WORK):
        shutil.rmtree(WORK)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK], check=True)
    SRC = os.path.join(WORK, "src")
else:
    # Kaggle datasets are read-only; the package just needs to be importable.
    candidate = os.path.join(SOURCE_DIR, "src")
    SRC = candidate if os.path.isdir(candidate) else SOURCE_DIR

sys.path.insert(0, SRC)
import video_memory
print("imported video_memory from:", os.path.dirname(video_memory.__file__))

## 4. Sanity check — run the CPU test batteries firstIf these fail, stop: the bug is in logic, not in the model wiring, and it is farcheaper to fix it locally than on a GPU box.

In [ ]:
TESTS = os.path.join(os.path.dirname(SRC), "tests")
if os.path.isdir(TESTS):
    for t in ("test_resolve.py", "test_cascade.py"):
        p = os.path.join(TESTS, t)
        if os.path.exists(p):
            print("==", t)
            subprocess.run([sys.executable, p], check=False)
else:
    print("tests/ not found next to src/ — skipping (fine if you uploaded src only)")

## 5. Get a videoAny `.mp4` works — the system does not depend on a particular dataset. Use aKaggle video dataset, upload your own clip, or fall back to the synthetic clipbelow (which exists to prove the pipeline runs end to end, **not** to producemeaningful numbers).When your Ego4D / X-LeBench license clears, point `VIDEO_PATH` at that footageand rerun this notebook unchanged.

In [ ]:
import numpy as np, cv2

def make_synthetic_clip(path="/kaggle/working/synthetic.mp4", seconds=60, fps=10):
    """A crude scripted clip: a coloured 'scene' that changes every 15s, with a
    moving square. Enough to exercise decode -> tag -> intervalize -> resolve."""
    h, w = 240, 320
    writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    scenes = [(30, 60, 90), (90, 60, 30), (40, 90, 60), (70, 70, 120)]
    for i in range(seconds * fps):
        t = i / fps
        bg = scenes[int(t // 15) % len(scenes)]
        frame = np.full((h, w, 3), bg, dtype=np.uint8)
        x = int((t * 25) % (w - 40))
        cv2.rectangle(frame, (x, 100), (x + 40, 140), (220, 220, 220), -1)
        writer.write(frame)
    writer.release()
    return path

if not VIDEO_PATH:
    VIDEO_PATH = make_synthetic_clip()
    print("using synthetic fallback clip")

print("video:", VIDEO_PATH, "| exists:", os.path.exists(VIDEO_PATH))

## 6. IngestStage 3 loads **lazily** — if no frame escalates, Qwen2-VL is never downloaded.Watch the reported `escalation_rate`: that is the cascade's efficiency claim.

In [ ]:
from video_memory.ingest import ingest_video

meta = ingest_video(
    video_path=VIDEO_PATH,
    out_path=OUT_PACK,
    sample_fps=SAMPLE_FPS,
    tau=TAU,
    max_frames=MAX_FRAMES,
    stage3_enabled=STAGE3,
    min_duration=MIN_DURATION,
)

import json
print(json.dumps(meta, indent=2))

## 7. Inspect the resolved memory

In [ ]:
from video_memory.pack import read_pack

pack = read_pack(OUT_PACK)
assertions = pack["assertions"]
print(f"{len(assertions)} resolved assertions\n")

def fmt(t):
    if t == float("inf"):
        return "  open "
    return f"{int(t // 60):02d}:{t % 60:05.2f}"

for a in sorted(assertions, key=lambda x: (x.relation, x.interval.start))[:40]:
    print(f"{fmt(a.interval.start)} -> {fmt(a.interval.end)}  "
          f"{a.relation:<11} {a.object:<20} conf={a.confidence:.2f}  "
          f"{a.interval.closure.value:<13} n_frames={len(a.evidence.frame_ids)}")

In [ ]:
# Sanity property: no two single-valued assertions may overlap.
from collections import defaultdict
from video_memory.perception.prompt_bank import exclusive_relations, DEFAULT_BANK

ex = exclusive_relations(DEFAULT_BANK)
slots = defaultdict(list)
for a in assertions:
    if a.relation in ex:
        slots[(a.subject, a.relation)].append(a)

violations = 0
for key, items in slots.items():
    items.sort(key=lambda x: x.interval.start)
    for p, n in zip(items, items[1:]):
        if n.interval.start < p.interval.end - 1e-9:
            violations += 1
            print("OVERLAP:", key, p.object, p.interval, "vs", n.object, n.interval)

print("single-valued overlap violations:", violations, "(must be 0)")

## 8. Coverage of the timelineHow much of the video does the memory actually make a claim about? Low coverageis not automatically bad — it means the system declined to assert things it wasunsure of — but it must be reported honestly alongside accuracy.

In [ ]:
duration = meta["video_duration_s"]
by_rel = defaultdict(float)
for a in assertions:
    end = min(a.interval.end, duration)
    by_rel[a.relation] += max(0.0, end - a.interval.start)

for rel, covered in sorted(by_rel.items()):
    print(f"{rel:<12} {covered:7.1f}s / {duration:.1f}s  ({100*covered/duration:5.1f}%)")

## 9. Download the packThe pack is small (embeddings dominate, and those are ~0.6 KB/frame). Commit itto the repo so the query side runs with no GPU and no dataset access.

In [ ]:
import shutil
archive = shutil.make_archive("/kaggle/working/pack", "zip", OUT_PACK)
print("wrote", archive, os.path.getsize(archive) / 1e6, "MB")
for f in sorted(os.listdir(OUT_PACK)):
    print(" ", f, os.path.getsize(os.path.join(OUT_PACK, f)) / 1e3, "KB")

## 10. Ablation: cascade off (SigLIP only)Rerun with `stage3_enabled=False` to get the A-noVLM arm. Comparing the twopacks is what turns "the cascade saves compute" into a measured number ratherthan a design claim.

In [ ]:
meta_no_vlm = ingest_video(
    video_path=VIDEO_PATH,
    out_path="/kaggle/working/pack_no_vlm",
    sample_fps=SAMPLE_FPS,
    tau=TAU,
    max_frames=MAX_FRAMES,
    stage3_enabled=False,
    min_duration=MIN_DURATION,
)

print("with VLM   :", meta["cascade"], f"x_realtime={meta['x_realtime']}")
print("without VLM:", meta_no_vlm["cascade"], f"x_realtime={meta_no_vlm['x_realtime']}")